# 05 Why MCP Exists

## MCP 解决的不是“能不能调函数”，而是能力系统如何被组织

一旦开始认真构建 Agent，很快就会发现一个现实问题：模型不是问题，单个工具也不是问题，真正变复杂的是外部能力怎么接、怎么描述、怎么治理、怎么复用。

在最早期的 demo 阶段，这个问题很容易被低估。因为那时工具少、上下文短、调用链简单，似乎只要写几个 function schema、塞几段文档、再手工拼一点 prompt，一个“会用工具的系统”就已经能跑起来。

但只要系统继续往前走，这套做法几乎一定会开始失真。工具定义变多之后，语义会分散；文档接入变多之后，资源入口会混乱；宿主环境一变，接入方式又要重写；调用链一长，调试和治理的成本会迅速上升。

MCP 的价值，就是在这个节点出现的。它解决的不是“模型能不能调用外部能力”这个最浅层的问题，而是：**外部能力如何以一种对模型、对 runtime、对宿主都更稳定的方式被组织起来。**

## 先给结论

这一章最重要的判断可以概括为一句话：

> MCP 存在的意义，不是把函数挂给模型，而是把原本零散、隐式、宿主绑定的外部能力，提升为一层标准化、可发现、可复用、可治理的能力接口。

这句话里有四个关键词很重要：

- `标准化`：不同能力用统一方式暴露
- `可发现`：系统能知道外面到底有什么可以用
- `可复用`：能力不是为单一应用写死的
- `可治理`：能力接入不是无限堆积的私有胶水代码

如果没有这四层视角，MCP 很容易被误读成“又一个 function calling 包装层”。这会把它真正解决的问题直接抹平。

## 1. 在 MCP 之前，问题到底出在哪里

很多人第一次接触 MCP 时会问：没有它之前，系统不是也能跑吗。这个问题很合理，答案也很直接：当然能跑，但能跑不等于结构清楚，也不等于可持续。

在 MCP 之前，典型系统通常靠几种方式接外部能力：

- 直接在应用代码里写死工具定义
- 手工把函数 schema 拼进模型调用参数
- 手工把文档片段注入 prompt
- 每个宿主各自维护一套外部能力接法

这些方式在 demo 阶段都有效，因为它们足够快。但问题也都很明显：

- 工具和文档是两套平行系统
- 能力描述散落在应用层代码里
- 模型能读到什么、能调用什么、如何发现，往往没有统一入口
- 一旦换一个宿主环境，很多能力定义又要重新接一遍

这说明问题的根并不在“没有函数调用能力”，而在“能力组织方式太临时”。

## 2. Function Calling 为什么不够，但并不是没用

这里需要一个比较克制的判断。Function calling 不是错误方案，也不是低级方案。相反，它是让模型从自由文本走向结构化动作的关键一步。没有这一层，很多 Agent 系统根本起不来。

但 function calling 解决的问题范围是有限的。它主要回答的是：

- 模型如何输出一个结构化动作意图
- 宿主如何把这个意图映射到某个函数执行

而它没有天然解决下面这些更系统的问题：

- 外部只读资源怎么统一暴露
- 文档、配置、知识片段怎么被发现和读取
- 任务模板怎么成为可复用入口
- 不同客户端怎样共享同一批能力
- 能力元信息如何从“应用私有代码”升级成“协议层描述”

所以，MCP 不是来否定 function calling，而是把 function calling 放回它应有的位置：它是能力执行的一部分，但不是整个能力治理体系。

## 3. 传统方案最大的问题之一：文档和函数是分裂的

很多 LLM 系统都有一个相同的结构问题：可执行能力和可读上下文分别存在，却没有统一接口。

比如，一个系统可能同时拥有：

- 一堆可调用函数
- 一些 API 文档
- 一些业务规则文档
- 一些本地配置、模板、知识片段

但在传统做法里，这些东西往往被分开处理：

- 函数通过 tool schema 暴露
- 文档通过 prompt 拼接或检索注入
- 模板通过应用代码硬编码

结果就是：系统虽然拥有能力，但没有统一能力面。模型对“能做什么”和“能看什么”的理解，依赖一堆分散的接线方式。

这也是为什么你前面那句“MCP 可以理解成把文档和函数写在一起”虽然不完全严格，但抓到了一个关键直觉。更准确地说，MCP 试图把“可读知识”和“可执行动作”放进同一套协议化暴露体系里，让模型宿主面对的是一个统一能力面，而不是很多种临时拼接方法。

In [ ]:
traditional_setup = {
    "tools": "写在应用层函数注册逻辑里",
    "docs": "被切片后手工塞进 prompt",
    "templates": "写在业务代码或字符串常量里",
    "problem": "能力存在，但入口分散，描述不统一，难以复用",
}

for key, value in traditional_setup.items():
    print(f"{key}: {value}")

## 4. Resource 入口缺失，是很多 Agent 系统隐形的结构短板

如果只盯着工具，很容易忽略另一个同样关键的问题：模型并不总是需要执行动作，很多时候它只是需要可靠地读取外部上下文。

例如：

- 读项目说明文档
- 读产品需求片段
- 读配置文件或接口规范
- 读知识库中的一段标准定义

这些都不是工具调用意义上的“执行”，但又远远不只是把一大坨文本提前塞进系统 prompt 就能优雅解决的问题。

传统方案通常有两种极端：

- 直接全量塞进上下文，结果噪音高、成本高、控制差
- 完全依赖应用侧自定义检索逻辑，结果每个项目一套私有约定

MCP 把 resource 作为第一等能力来看待，本质上就是在说：**外部可读信息不应该只是 prompt 材料，而应该成为协议层可被发现、可被读取、可被治理的对象。**

## 5. Prompt 模板也应该被看成能力，而不是散落的字符串

除了工具和资源，还有一种东西在传统系统里常常被低估，那就是 prompt 模板本身。

很多实际任务不是靠一个通用 system prompt 就能完成，而是需要一些可复用的任务入口：

- 针对文档分析的标准任务模板
- 针对需求拆解的结构化提问模板
- 针对代码审阅或方案生成的固定入口形式

如果这些模板只是散落在业务代码中的字符串常量，那么它们对模型来说并没有成为“可发现能力”，对宿主来说也不具备可组合性。

把 prompt 也纳入能力层，不是为了把一切都协议化到形式主义，而是为了让任务入口本身也成为系统可管理的一部分。这个点很容易被忽略，但对于真正想做可复用 Agent 平台的人来说，它并不次要。

## 6. 传统接法最大的问题之一，是宿主绑定太重

在没有统一协议的时候，外部能力往往和当前应用宿主绑得很死。你在一个 notebook、一个桌面应用、一个编辑器插件里各自接一套工具，看起来都能跑，但本质上是在复制同一个问题：

- 每个宿主都自己知道工具在哪
- 每个宿主都自己维护文档怎么读
- 每个宿主都自己定义 prompt 模板怎么暴露

这种方式短期内最直接，但长期代价很高。因为一旦能力不再是少量私有功能，而是希望被多个宿主共享、被多个 Agent 复用、被多个任务组合，宿主绑定就会成为最大的扩展阻力。

MCP 的一个核心价值，就是把“能力在哪里”和“宿主如何用它”之间加了一层标准接口。这样，能力提供方可以稳定暴露能力，宿主只需要学会接协议，而不必为每个能力源都写一套私有适配。

## 7. 可发现性比很多人以为的更重要

如果一个能力存在，但系统不知道它存在、也不知道怎么描述它，那它在运行时意义上就几乎等于不存在。

可发现性之所以重要，是因为 Agent 不是写死单条流程的脚本。它需要在某个时刻知道：

- 现在可用哪些工具
- 有哪些资源可以读取
- 有哪些现成 prompt 模板可以作为任务入口

传统方案里，这些信息通常隐含在应用代码里。应用作者当然知道，但模型宿主、其他客户端、甚至后来的维护者都未必知道。MCP 把这种隐式知识拉到协议面上，本质上是在做能力 discoverability。

没有 discoverability，系统只能依赖预设接线；有了 discoverability，系统才开始具备能力层的可扩展性。

In [ ]:
mcp_style_surface = {
    "tools": ["search_docs", "run_sql", "get_weather_forecast"],
    "resources": ["product://requirements", "repo://architecture", "team://playbook"],
    "prompts": ["summarize_prd", "map_candidate_to_jd"],
}

for capability_type, items in mcp_style_surface.items():
    print(f"{capability_type}: {items}")

## 8. MCP 的本质价值之一，是把“接能力”升级成“治理能力”

系统规模一旦上来，真正难的从来不是再加一个工具，而是加完之后整个系统还能不能保持清晰。

治理问题会在很多地方出现：

- 工具命名是否一致
- 资源是否有稳定 URI 或等价标识
- 提示模板是否能被版本化管理
- 不同宿主接入后是否看到同样的能力描述
- 工具、资源、模板之间是否能形成统一能力面

如果没有协议层，治理几乎只能在应用私有约定里完成；这意味着每个团队、每个项目、每个宿主都在重复造自己的半套规范。MCP 把这件事往前提了一层：先约定能力怎么暴露，再谈不同应用怎么消费。这个顺序的变化，就是它的工程价值。

## 9. Agent 与 MCP 的职责边界，必须说清楚

MCP 越被讨论，越容易出现两种相反的误解：

- 一种误解是把它贬成“function calling 换皮”
- 另一种误解是把它抬成“有了 MCP 就有 Agent”

这两种看法都不对。

更准确的职责边界应该是：

- Agent 负责目标、状态、策略、循环和终止
- MCP 负责以标准化方式暴露 tools、resources、prompts
- 模型负责在上下文中生成下一步意图

也就是说，MCP 是能力层，Agent 是任务层。没有 MCP，Agent 仍然可以存在，但能力接入会越来越零散；没有 Agent，MCP 仍然可以存在，但只会成为被动能力目录，而不会自己推进任务。

## 10. 为什么在 Agent 时代，MCP 变得更有必要

如果系统只是单轮问答，很多能力问题都可以靠手工拼接勉强解决。但 Agent 的出现改变了事情的难度级别。

因为 Agent 不是只调用一次外部函数，而是可能：

- 先读资源再选工具
- 连续读取多个资源交叉验证
- 在失败后改走另一条能力路径
- 在不同任务间复用同一批能力入口

一旦任务开始以这种方式组织，外部能力就不再是几个点状插件，而变成了 Agent Runtime 持续依赖的一层操作面。MCP 在这个阶段的意义就会被放大，因为它让 Agent 面对的是一个结构化能力面，而不是一堆分散接口。

## 11. MCP 不是什么

为了防止概念继续漂移，也有必要反过来说清楚 MCP 不是什么。

MCP 不是：

- 一个替代模型推理的算法
- 一个自动规划任务的 Agent 框架
- 一个保证模型一定正确调工具的魔法层
- 一个只适合某个单一宿主的私有插件格式

它更像是模型宿主与外部能力提供方之间的一套共同语言。这个共同语言本身不会替你做架构决策，但它会显著降低架构决策被一堆私有胶水代码拖垮的概率。

## 12. 从展示角度看，为什么这一层值得专门写出来

如果这个项目只是写给工程师自己看，完全可以直接堆代码。但既然它也面向招聘场景，就更需要把 MCP 的意义讲到一个非实现细节层面。

因为它体现的不是“我知道一个新协议”，而是另一种更重要的判断力：

- 你能看出系统复杂度真正出现在哪里
- 你知道哪些问题是 demo 阶段看不出来、扩展阶段一定会爆出来的
- 你理解能力接入不是函数多不多的问题，而是组织方式的问题

从这个角度看，MCP 不是一个炫技点，而是你对 Agent 系统复杂度有结构化认知的证据。

## 13. 从“为什么需要 MCP”走向“它具体如何组织能力”

到这一章结束，真正应该建立起来的不是对某个协议名字的好感，而是一种判断：

- Agent 一旦需要稳定调度外部能力，能力层就必须被单独设计
- 传统 function calling 和 prompt 拼接能起步，但很难承担长期治理
- 工具、资源、模板如果不能形成统一能力面，系统会越来越依赖私有胶水代码

在这个判断建立之后，下一章再去看 MCP 的具体组织方式，例如 tool、resource、prompt 各自扮演什么角色，host、client、server 如何协作，讨论才真正有意义。否则，协议细节会显得像概念灌输，而不是问题自然推导出来的结构答案。

## 14. 本章结论

这一章可以压缩成下面这些关键判断：

- MCP 解决的不是“模型能不能调函数”，而是外部能力如何被组织成统一能力面。
- Function calling 很重要，但它只解决动作输出的一部分问题，不等于完整能力治理。
- 工具、文档、资源、模板在传统系统里往往是分裂的，MCP 试图把它们协议化地拉回同一层。
- Resource 和 prompt 之所以重要，是因为 Agent 需要的从来不只是执行动作，还包括读取上下文和复用任务入口。
- MCP 是能力层，Agent 是任务层；前者不替代后者，后者也不应无视前者。

下一章会进入 MCP 的具体组织结构，不再停留在“为什么需要”这一层，而是开始拆它到底如何把 tools、resources、prompts 组织成一个可被 Agent Runtime 消费的标准化能力系统。